In [ ]:
# Cellule 1 — installation, chargement et contrôles d'entrée
!pip -q install "transformers>=4.40,<5" sentencepiece

import hashlib, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ---------- paramètres (ne pas modifier sans raison) ----------
MODEL_NAME = "facebook/esm2_t33_650M_UR50D"
MAX_RESIDUES = 1022
EXPECTED_DIM = 1280
COSINE_THRESHOLD = 0.999
REQUIRE_GPU = True

from google.colab import drive
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/files/data/colab_control")
SPECIES_KEY = "oryza_sativa"  # Set to zea_mays, nicotiana_tabacum_k326, or nicotiana_tabacum_tn90.
PREFIX = f"{SPECIES_KEY}_embedding_control_200"
FASTA = BASE / f"{PREFIX}.fasta"
META = BASE / f"{PREFIX}.csv"
STORED = BASE / f"{PREFIX}.npy"
OUT_JSON = BASE / f"{PREFIX}_results.json"
OUT_CSV = BASE / f"{PREFIX}_per_protein.csv"
OUT_VARIANTS = BASE / f"{PREFIX}_variants.json"

for path in (FASTA, META, STORED):
    assert path.exists(), f"Fichier absent : {path}"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_fasta(path):
    records, accession, parts = {}, None, []
    with open(path) as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if accession is not None:
                    records[accession] = "".join(parts)
                accession = line[1:].split()[0]
                assert accession not in records, f"Accession dupliquée : {accession}"
                parts = []
            else:
                parts.append(line)
    if accession is not None:
        records[accession] = "".join(parts)
    return records

# ---------- environnement ----------
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
elif REQUIRE_GPU:
    raise RuntimeError(
        "Pas de GPU : menu Exécution > Modifier le type d'exécution > GPU (T4), "
        "puis relance. (REQUIRE_GPU=False pour forcer le CPU, très lent.)"
    )

# ---------- métadonnées et vecteurs stockés ----------
meta = pd.read_csv(META)
missing_columns = {"protein_accession", "group"} - set(meta.columns)
assert not missing_columns, f"Colonnes manquantes : {missing_columns}"
assert meta["protein_accession"].is_unique, "Accessions dupliquées dans le CSV"

stored = np.load(STORED).astype(np.float32)
assert stored.shape == (len(meta), EXPECTED_DIM), stored.shape
assert np.isfinite(stored).all(), "NaN ou infini dans les vecteurs stockés"
assert (np.linalg.norm(stored, axis=1) > 0).all(), "Vecteur stocké nul"

# ---------- séquences ----------
sequences = {k: v.replace(" ", "").upper() for k, v in read_fasta(FASTA).items()}
assert set(meta["protein_accession"]) == set(sequences), (
    "FASTA et CSV ne contiennent pas les mêmes accessions : "
    f"seulement CSV={sorted(set(meta['protein_accession']) - set(sequences))[:5]}, "
    f"seulement FASTA={sorted(set(sequences) - set(meta['protein_accession']))[:5]}"
)
bad = [a for a, s in sequences.items() if not s or not re.fullmatch(r"[A-Z]+", s)]
assert not bad, f"Séquences vides ou à caractères invalides : {bad[:5]}"

accessions = meta["protein_accession"].tolist()
seq_md5 = {a: hashlib.md5(sequences[a].encode()).hexdigest() for a in accessions}
lengths = {a: len(sequences[a]) for a in accessions}

input_hashes = {
    "fasta_sha256": sha256_file(FASTA),
    "csv_sha256": sha256_file(META),
    "npy_sha256": sha256_file(STORED),
}

print("\nÉchantillon :", len(meta))
print("Groupes :", meta["group"].value_counts().to_dict())
print("Séquences > %d aa (tronquées) :" % MAX_RESIDUES,
      sum(1 for n in lengths.values() if n > MAX_RESIDUES))
print("Embeddings stockés :", stored.shape)
for name, value in input_hashes.items():
    print(f"{name}: {value}")


In [ ]:
# Cellule 2 — modèle ESM-2 et recalcul (méthode de référence)
import transformers
from transformers import AutoTokenizer, EsmModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()  # l'avertissement sur le "pooler" est sans effet

try:
    from huggingface_hub import HfApi
    MODEL_REVISION = HfApi().model_info(MODEL_NAME).sha
except Exception:
    MODEL_REVISION = None
print("Révision du modèle :", MODEL_REVISION)

def _embed_batch(batch, pooling, fp16):
    clean = [seq[:MAX_RESIDUES] for seq in batch]
    enc = tokenizer(
        clean,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_RESIDUES + 2,
    )
    enc = {key: value.to(DEVICE) for key, value in enc.items()}

    with torch.inference_mode():
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=bool(fp16 and DEVICE.type == "cuda"),
        ):
            hidden = model(**enc).last_hidden_state
    hidden = hidden.float()

    mask = enc["attention_mask"].bool()
    if pooling == "residues":            # référence : sans jetons spéciaux
        special = torch.zeros_like(mask)
        for special_id in tokenizer.all_special_ids:
            special |= enc["input_ids"] == special_id
        keep = mask & ~special
    elif pooling == "all_tokens":        # variante de diagnostic
        keep = mask
    else:
        raise ValueError(pooling)

    keep = keep.unsqueeze(-1).to(hidden.dtype)
    pooled = (hidden * keep).sum(dim=1) / keep.sum(dim=1).clamp(min=1)
    return pooled.cpu().numpy()

def embed_sequences(seqs, pooling="residues", fp16=False, batch_size=8):
    """Embeddings dans l'ordre d'entrée ; tri par longueur en interne, repli si mémoire insuffisante."""
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]), reverse=True)
    out = np.zeros((len(seqs), EXPECTED_DIM), dtype=np.float32)
    position, size = 0, batch_size if DEVICE.type == "cuda" else 1
    while position < len(order):
        idx = order[position:position + size]
        try:
            out[idx] = _embed_batch([seqs[i] for i in idx], pooling, fp16)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if size == 1:
                raise
            size = max(1, size // 2)
            print("Mémoire GPU insuffisante, taille de lot ->", size)
            continue
        position += len(idx)
        if position % 40 < len(idx) or position == len(order):
            print(f"{position}/{len(order)}")
    return out

ordered_seqs = [sequences[a] for a in accessions]
recomputed = embed_sequences(ordered_seqs, pooling="residues", fp16=False)

assert recomputed.shape == stored.shape, recomputed.shape
assert np.isfinite(recomputed).all(), "NaN ou infini dans les embeddings recalculés"
print("Embeddings recalculés :", recomputed.shape)


In [ ]:
# Cellule 3 — cosinus, verdict et rapports
def cosine_rows(a, b):
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return np.sum(a * b, axis=1)

meta = meta.copy()
meta["length"] = [lengths[a] for a in accessions]
meta["truncated"] = meta["length"] > MAX_RESIDUES
meta["sequence_md5"] = [seq_md5[a] for a in accessions]
meta["cosine"] = cosine_rows(recomputed, stored)
meta["passes"] = meta["cosine"] >= COSINE_THRESHOLD

def describe(frame):
    failed = frame.loc[~frame["passes"], ["protein_accession", "cosine", "length"]]
    return {
        "n": int(len(frame)),
        "minimum": float(frame["cosine"].min()),
        "median": float(frame["cosine"].median()),
        "mean": float(frame["cosine"].mean()),
        "share_passing": float(frame["passes"].mean()),
        "below_threshold": failed.sort_values("cosine").to_dict("records"),
    }

groups = {name: describe(frame) for name, frame in meta.groupby("group", sort=True)}
truncation = {
    "truncated": describe(meta[meta["truncated"]]) if meta["truncated"].any() else None,
    "not_truncated": describe(meta[~meta["truncated"]]),
}

n = int(len(meta))
all_pass = bool(meta["passes"].all())
failing = int((~meta["passes"]).sum())
verdict = {
    "all_above_threshold": all_pass,
    "failing_count": failing,
    "md5_fill_allowed": all_pass,
    # borne supérieure à 95 % du taux de discordance si aucune discordance sur n contrôles
    "mismatch_rate_upper_95pct": (1 - 0.05 ** (1 / n)) if all_pass else None,
    "next_step": (
        "Remplir les MD5 des embeddings à partir de dim_sequence, marqués 'vérifiés par échantillonnage'."
        if all_pass else
        "NE PAS remplir les MD5. Lancer la cellule 4 (variantes de méthode) avant toute conclusion."
    ),
}

report = {
    "species_key": SPECIES_KEY,
    "model": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "max_residues": MAX_RESIDUES,
    "pooling": "mean over residue tokens, excluding special tokens, fp32",
    "cosine_threshold": COSINE_THRESHOLD,
    "input_files": input_hashes,
    "n_total": n,
    "global": describe(meta),
    "groups": groups,
    "truncation": truncation,
    "verdict": verdict,
}

with open(OUT_JSON, "w") as handle:
    json.dump(report, handle, indent=2)
meta[["protein_accession", "group", "length", "truncated",
      "sequence_md5", "cosine", "passes"]].to_csv(OUT_CSV, index=False)

display(pd.DataFrame([
    {"group": name, "n": v["n"], "minimum": v["minimum"], "median": v["median"],
     "share_passing": v["share_passing"], "below_threshold": len(v["below_threshold"])}
    for name, v in groups.items()
]))
print("\nVerdict :", "OK" if all_pass else f"ÉCHEC ({failing} protéine(s) sous {COSINE_THRESHOLD})")
print(verdict["next_step"])
if all_pass:
    print(f"Borne supérieure (95 %) du taux de discordance : {verdict['mismatch_rate_upper_95pct']:.2%}")
else:
    display(meta.loc[~meta["passes"],
                     ["protein_accession", "group", "length", "truncated", "cosine"]]
            .sort_values("cosine"))
print("\nRapports :", OUT_JSON, "et", OUT_CSV)


In [ ]:
# Cellule 4 — diagnostic (utile seulement si la cellule 3 signale des échecs)
if meta["passes"].all():
    print("Aucun échec : cellule sans objet.")
else:
    variants = {
        "all_tokens":      dict(pooling="all_tokens", fp16=False),
        "residues_fp16":   dict(pooling="residues",   fp16=True),
        "all_tokens_fp16": dict(pooling="all_tokens", fp16=True),
    }
    results = {}
    for name, options in variants.items():
        print("Variante :", name)
        vectors = embed_sequences(ordered_seqs, **options)
        cos = cosine_rows(vectors, stored)
        results[name] = {
            "minimum": float(cos.min()),
            "median": float(np.median(cos)),
            "failing_count": int((cos < COSINE_THRESHOLD).sum()),
            "recovers_all": bool((cos >= COSINE_THRESHOLD).all()),
        }
    with open(OUT_VARIANTS, "w") as handle:
        json.dump(results, handle, indent=2)
    display(pd.DataFrame(results).T)
    print("""
Lecture :
- une variante qui rétablit tout l'échantillon -> les anciens vecteurs ont été faits avec cette méthode ;
  les séquences sont bonnes, il faut aligner la méthode du pipeline ;
- aucune variante ne rétablit les échecs, et ils touchent des protéines non tronquées
  -> les anciens vecteurs viennent d'autres séquences : ré-embedder tout le riz ;
- échecs limités aux protéines tronquées -> écart de méthode de troncature.
""")
